# Frobenius Motif Segmentation — Parameter Tuning

Interactive notebook for tuning SAM segmentation parameters on individual panel crops.
Adjust sliders, hit **Run segmentation**, inspect the annotated image and detection table,
then copy the final values back to `panel_art/motif_segment.py`.

---

## Setup instructions

### Local (recommended — faster iteration, no upload needed)

```bash
# From the repo root
cd /path/to/african-artifacts

# Install Jupyter into the existing UV environment
uv pip install --project src/python jupyter ipywidgets

# Launch — the kernel will use the project venv automatically
uv run --project src/python jupyter notebook src/python/motif_tuning.ipynb
```

The panel crops in `frobenius_artifacts/analysis/panels/` are loaded from disk;
the SAM checkpoint is read from `src/python/sam_vit_b_01ec64.pth`.
Inference runs on CPU (~30–90 s per panel on Intel Mac).

### Google Colab

1. Upload this notebook to Colab (`File → Upload notebook`).
2. Upload your panel crop PNGs to the Colab session storage
   (`Files` panel → upload, or mount Google Drive).
3. Run **Cell 1 (Colab setup)** — it installs dependencies and downloads the
   SAM checkpoint (~370 MB, takes ~1 min on Colab).
4. Set `PANEL_DIR` in Cell 2 to wherever you uploaded the crops.
5. Run all remaining cells.

Colab with a T4 GPU runs each segmentation in ~3–5 s.

In [2]:
# ── Cell 1: environment setup (Colab only — skip locally) ──────────────────
import sys
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    print("Colab detected — installing dependencies...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "segment-anything",
        "opencv-python-headless",
        "Pillow",
        "numpy>=1.24,<2",
        "ipywidgets",
    ], check=True)

    # Download SAM ViT-B checkpoint if not already present
    import os
    CKPT = "sam_vit_b_01ec64.pth"
    if not os.path.exists(CKPT):
        print("Downloading SAM checkpoint (~370 MB)...")
        subprocess.run(["wget", "-q",
            "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
        ], check=True)
        print("Done.")
    else:
        print("Checkpoint already present.")
else:
    print("Local environment — skipping Colab setup.")

Local environment — skipping Colab setup.


In [3]:
# ── Cell 2: imports & paths ────────────────────────────────────────────────
import os, sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Path configuration ────────────────────────────────────────────────────
# Adjust these if running on Colab or from a different working directory.

ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    # Point to wherever you uploaded the panel crops
    PANEL_DIR  = Path("/content/panels")
    CHECKPOINT = Path("sam_vit_b_01ec64.pth")
    # panel_art package is not installed on Colab — import helpers inline below
    PANEL_ART_AVAILABLE = False
else:
    # Repo-relative paths (notebook lives at src/python/)
    REPO_ROOT  = Path("../..")
    PANEL_DIR  = REPO_ROOT / "frobenius_artifacts/analysis/panels"
    CHECKPOINT = Path("sam_vit_b_01ec64.pth")  # src/python/sam_vit_b_01ec64.pth

    # Add package to path so we can import panel_art directly
    sys.path.insert(0, str(Path(".").resolve()))
    try:
        from panel_art.motif_segment import (
            filter_and_nms, classify_scale, annotate_detections,
            load_generator, segment_panel, Detection,
            DEFAULT_IOU_THRESH, DEFAULT_STABILITY_THRESH,
            DEFAULT_NMS_IOU, DEFAULT_MIN_AREA, DEFAULT_MAX_AREA,
            DEFAULT_POINTS_PER_SIDE, DEFAULT_MAX_ASPECT,
        )
        PANEL_ART_AVAILABLE = True
        print("panel_art package loaded.")
    except ImportError as e:
        print(f"panel_art not importable ({e}) — using inline helpers.")
        PANEL_ART_AVAILABLE = False

# Discover panel crops
panel_files = sorted(PANEL_DIR.glob("*_panel_*.png")) if PANEL_DIR.exists() else []
print(f"{len(panel_files)} panel crop(s) found in {PANEL_DIR}")
if not panel_files:
    print("WARNING: no panels found. Check PANEL_DIR above.")


panel_art package loaded.
62 panel crop(s) found in ../../frobenius_artifacts/analysis/panels


In [4]:
# ── Cell 3: inline helpers (used when panel_art is not importable) ─────────
# These mirror panel_art/motif_segment.py exactly so the notebook works
# standalone on Colab. When panel_art IS available they are overwritten below.

def _iou(a, b):
    ax1, ay1 = a[0], a[1]
    ax2, ay2 = a[0]+a[2], a[1]+a[3]
    bx1, by1 = b[0], b[1]
    bx2, by2 = b[0]+b[2], b[1]+b[3]
    inter_w = max(0, min(ax2,bx2) - max(ax1,bx1))
    inter_h = max(0, min(ay2,by2) - max(ay1,by1))
    inter = inter_w * inter_h
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter/union if union > 0 else 0.0

def _filter_and_nms_inline(
    masks, img_area,
    min_area=0.01, max_area=0.85,
    iou_thresh=0.70, stability_thresh=0.75,
    nms_iou=0.40, max_aspect=7.0,
):
    kept = []
    for m in masks:
        x, y, w, h = m["bbox"]
        area_ratio = m["area"] / img_area
        if area_ratio < min_area or area_ratio > max_area: continue
        if w == 0 or h == 0: continue
        if max(w,h)/max(min(w,h),1) > max_aspect: continue
        if m["predicted_iou"] < iou_thresh: continue
        if m["stability_score"] < stability_thresh: continue
        kept.append(m)
    kept.sort(key=lambda m: m["area"], reverse=True)
    final, suppressed = [], set()
    for i, m in enumerate(kept):
        if i in suppressed: continue
        final.append(m)
        for j in range(i+1, len(kept)):
            if j not in suppressed and _iou(m["bbox"], kept[j]["bbox"]) > nms_iou:
                suppressed.add(j)
    return final

def _classify_scale_inline(area_ratio):
    return "register" if area_ratio > 0.25 else "motif"

# Use package versions when available; inline versions otherwise
if not PANEL_ART_AVAILABLE:
    filter_and_nms   = _filter_and_nms_inline
    classify_scale   = _classify_scale_inline
    DEFAULT_IOU_THRESH       = 0.70
    DEFAULT_STABILITY_THRESH = 0.75
    DEFAULT_NMS_IOU          = 0.40
    DEFAULT_MIN_AREA         = 0.01
    DEFAULT_MAX_AREA         = 0.85
    DEFAULT_POINTS_PER_SIDE  = 32
    DEFAULT_MAX_ASPECT       = 7.0

print("Helpers ready.")


Helpers ready.


In [5]:
# ── Cell 4: load SAM (run once per session) ────────────────────────────────
import torch
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

def _resolve_device():
    if torch.cuda.is_available(): return "cuda"
    return "cpu"  # MPS excluded: SAM-1 uses float64, MPS can't handle it

# Cache so re-running this cell is instant
_sam_cache = {}

def get_generator(points_per_side: int, iou_thresh: float, stability_thresh: float):
    """Load (or retrieve cached) SAM generator. Reloads only if points_per_side changes."""
    key = points_per_side
    if key not in _sam_cache:
        device = _resolve_device()
        print(f"Loading SAM ViT-B on {device} (points_per_side={points_per_side})...")
        sam = sam_model_registry["vit_b"](checkpoint=str(CHECKPOINT))
        sam.to(device=device)
        _sam_cache[key] = (sam, device)
        print("Done.")
    sam, device = _sam_cache[key]
    gen = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=points_per_side,
        pred_iou_thresh=iou_thresh,
        stability_score_thresh=stability_thresh,
        min_mask_region_area=200,
    )
    return gen

# Pre-load with current defaults so the first Run is instant
_ = get_generator(DEFAULT_POINTS_PER_SIDE, DEFAULT_IOU_THRESH, DEFAULT_STABILITY_THRESH)
print("SAM ready — proceed to the tuning cell.")

Loading SAM ViT-B on cpu (points_per_side=32)...
Done.
SAM ready — proceed to the tuning cell.


In [6]:
# ── Cell 5: annotation helper (matplotlib, no file write) ─────────────────
SCALE_COLOURS = {
    "register": (1.0, 0.31, 0.31),  # red
    "motif":    (0.31, 0.78, 0.31), # green
}

def draw_detections(img_rgb: np.ndarray, detections: list, ax):
    """Draw bounding boxes + labels on a matplotlib axes."""
    ax.imshow(img_rgb, cmap="gray" if img_rgb.ndim == 2 else None)
    h, w = img_rgb.shape[:2]
    font_size = max(5, min(w, h) / 80)

    for d in detections:
        if PANEL_ART_AVAILABLE:
            bx, by, bw, bh = d.bbox["x"], d.bbox["y"], d.bbox["w"], d.bbox["h"]
            scale = d.scale
            area  = d.area_ratio
            idx   = d.index
        else:
            bx, by, bw, bh = d["bbox"]
            area  = d["area_ratio"]
            scale = d["scale"]
            idx   = d["index"]

        colour = SCALE_COLOURS.get(scale, (0.8, 0.8, 0.8))
        rect = mpatches.Rectangle(
            (bx, by), bw, bh,
            linewidth=max(1, min(w,h)//300),
            edgecolor=colour, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            bx + 2, by - 3,
            f"#{idx} {scale} {area*100:.1f}%",
            fontsize=font_size, color="white",
            bbox=dict(facecolor=colour, edgecolor="none", pad=1, alpha=0.85),
            va="bottom",
        )

    # Legend
    legend_handles = [
        mpatches.Patch(color=c, label=s)
        for s, c in SCALE_COLOURS.items()
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=font_size)
    ax.axis("off")

In [7]:
from pathlib import Path

# ── Cell 6: interactive tuning UI ──────────────────────────────────────────

# ── Panel selector ────────────────────────────────────────────────────────
default_panels = set([
    "EBA-B_00425_Ibadan_q97912_i1_panel_0_cropped.png",
    "EBA-Div_00303_Ado_Ekiti_q166559_i1_panel_00.png",
    "EBA-Div_00311_Ife_q166566_i1_panel_00.png",
    "EBA-Div_00311_Ife_q166566_i1_panel_01.png",
    "EBA-Div_00311_Ife_q166566_i1_panel_02.png",
    "EBA-Div_00312_Ife_q166567_i1_panel_00.png" ,
    "EBA-Div_00312_Ife_q166567_i1_panel_01.png"   ,
    "FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_00.png",
    "FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_01.png",
    "FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_02.png",
    "FoA_04-5585_Modakeke_(Ife)_q48635_i1_panel_00.png",
])
default_panel_selections = [str(p) for p in panel_files if p.name in default_panels]
print(default_panel_selections, str(panel_files[0]))
panel_picker = widgets.SelectMultiple(
    options=[(p.name, str(p)) for p in panel_files],
    value=default_panel_selections if panel_files else [],
    rows=8,
    description="Panels:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "60px"},
)
w_sel_count = widgets.HTML(value="")

out_preview = widgets.Output()

def _show_previews(paths):
    out_preview.clear_output(wait=True)
    w_sel_count.value = f"<i style='color:#666'>{len(paths)} selected — Ctrl/⌘+click to add, Shift+click to range-select</i>"
    if not paths:
        return
    with out_preview:
        n = len(paths)
        thumb_w = min(3.5, 18 / max(n, 1))
        fig, axes = plt.subplots(1, n, figsize=(thumb_w * n, 3.5), squeeze=False)
        for ax, p in zip(axes[0], paths):
            img = Image.open(p).convert("RGB")
            img.thumbnail((300, 380))
            ax.imshow(np.array(img))
            ax.set_title(Path(p).name, fontsize=6)
            ax.axis("off")
        plt.tight_layout(pad=0.3)
        plt.show()

def _on_picker_change(change):
    _show_previews(list(change["new"]))

panel_picker.observe(_on_picker_change, names="value")
_show_previews(list(panel_picker.value))

# ── Parameter widgets ─────────────────────────────────────────────────────
_sl = dict(continuous_update=False, style={"description_width": "120px"},
           layout=widgets.Layout(width="90%"))

w_points = widgets.SelectionSlider(
    options=[8, 16, 32, 64],
    value=DEFAULT_POINTS_PER_SIDE,
    description="points/side",
    **_sl,
)
w_iou_thresh = widgets.FloatSlider(
    min=0.40, max=0.95, step=0.01, value=DEFAULT_IOU_THRESH,
    description="iou_thresh",
    readout_format=".2f",
    **_sl,
)
w_stab = widgets.FloatSlider(
    min=0.40, max=0.95, step=0.01, value=DEFAULT_STABILITY_THRESH,
    description="stability",
    readout_format=".2f",
    **_sl,
)
# Area sliders display in percent (e.g. 1.00 = 1% of panel area).
# Divided by 100 when passed to filter_and_nms.
w_min_area = widgets.FloatSlider(
    min=0.05, max=15.0, step=0.05, value=DEFAULT_MIN_AREA * 100,
    description="min_area %",
    readout_format=".2f",
    **_sl,
)
w_max_area = widgets.FloatSlider(
    min=30.0, max=100.0, step=1.0, value=DEFAULT_MAX_AREA * 100,
    description="max_area %",
    readout_format=".0f",
    **_sl,
)
w_nms_iou = widgets.FloatSlider(
    min=0.05, max=0.70, step=0.01, value=DEFAULT_NMS_IOU,
    description="nms_iou",
    readout_format=".2f",
    **_sl,
)
w_max_aspect = widgets.FloatSlider(
    min=1.5, max=15.0, step=0.5, value=DEFAULT_MAX_ASPECT,
    description="max_aspect",
    readout_format=".1f",
    **_sl,
)
w_show_raw = widgets.Checkbox(
    value=False,
    description="Show raw SAM masks (all, before filtering — useful when you get 0 detections)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

run_btn    = widgets.Button(description="Run segmentation", button_style="primary",
                            layout=widgets.Layout(width="180px", height="36px"))
out_status = widgets.Output()
out_image  = widgets.Output()
out_table  = widgets.Output()
out_export = widgets.Output()
out_raw    = widgets.Output()

# ── Layout ────────────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HTML("<b>SAM generator</b> — changing <code>points/side</code> triggers a model reload:"),
    w_points,
    widgets.HTML("<b>Filter parameters:</b>"),
    w_iou_thresh, w_stab, w_min_area, w_max_area, w_nms_iou, w_max_aspect,
])

display(
    widgets.HTML("<h3 style='margin-bottom:4px'>SAM Parameter Tuning</h3>"),
    panel_picker,
    w_sel_count,
    out_preview,
    controls,
    w_show_raw,
    run_btn,
    out_status,
    out_image,
    out_table,
    out_export,
    out_raw,
)

# Global state
latest_detections = None # set after SAM runs

# ── Run callback ──────────────────────────────────────────────────────────
def on_run(_):
    out_status.clear_output(wait=True)
    out_image.clear_output(wait=True)
    out_table.clear_output(wait=True)
    out_export.clear_output(wait=True)
    out_raw.clear_output(wait=True)

    selected = [Path(p) for p in panel_picker.value]
    if not selected:
        with out_status:
            print("No panels selected — Ctrl/⌘+click to select one or more from the list.")
        return

    pts   = int(w_points.value)
    iou_t = float(w_iou_thresh.value)
    stab  = float(w_stab.value)
    mina  = float(w_min_area.value) / 100   # widget shows %, filter expects fraction
    maxa  = float(w_max_area.value) / 100
    nms   = float(w_nms_iou.value)
    masp  = float(w_max_aspect.value)

    with out_status:
        print(f"Loading SAM (pts/side={pts})...")
    generator = get_generator(pts, iou_t, stab)
    with out_status:
        print(f"Running {len(selected)} panel(s)...\n")

    for panel_path in selected:
        with out_status:
            print(f"▶ {panel_path.name}")

        img_pil  = Image.open(panel_path).convert("RGB")
        img_np   = np.array(img_pil)
        h, w     = img_np.shape[:2]
        img_area = h * w

        raw_masks = generator.generate(img_np)

        # Filter breakdown
        n_too_small  = sum(1 for m in raw_masks if m["area"] / img_area < mina)
        n_too_large  = sum(1 for m in raw_masks if m["area"] / img_area > maxa)
        n_bad_aspect = sum(1 for m in raw_masks
                           if (lambda x,y,bw,bh: bw>0 and bh>0 and
                               max(bw,bh)/max(min(bw,bh),1) > masp)(*m["bbox"]))
        n_low_iou    = sum(1 for m in raw_masks if m["predicted_iou"] < iou_t)
        n_low_stab   = sum(1 for m in raw_masks if m["stability_score"] < stab)

        with out_status:
            print(f"  {len(raw_masks)} raw masks:")
            print(f"    area < {mina*100:.2f}% (too small)  : {n_too_small}")
            print(f"    area > {maxa*100:.1f}%  (too large)  : {n_too_large}")
            print(f"    aspect > {masp:.1f}  (too thin)      : {n_bad_aspect}")
            print(f"    pred_iou < {iou_t:.2f}               : {n_low_iou}")
            print(f"    stability < {stab:.2f}               : {n_low_stab}")

        # Filter + NMS
        kept = filter_and_nms(
            raw_masks, img_area,
            min_area=mina, max_area=maxa,
            iou_thresh=iou_t, stability_thresh=stab,
            nms_iou=nms, max_aspect=masp,
        )
        detections = []
        # detection_idx_to_panel = {}
        for idx, m in enumerate(kept):
            x, y, bw, bh = [int(v) for v in m["bbox"]]
            ar = m["area"] / img_area
            if PANEL_ART_AVAILABLE:
                detections.append(Detection(
                    index=idx, bbox={"x": x, "y": y, "w": bw, "h": bh},
                    scale=classify_scale(ar), area_ratio=ar,
                    predicted_iou=float(m["predicted_iou"]),
                    stability_score=float(m["stability_score"]),
                ))
            else:
                detections.append({
                    "index": idx, "bbox": [x, y, bw, bh],
                    "scale": classify_scale(ar), "area_ratio": ar,
                    "predicted_iou": float(m["predicted_iou"]),
                    "stability_score": float(m["stability_score"]),
                })
            # detection_idx_to_panel[idx] = panel_path
        detections.sort(
            key=(lambda d: d.area_ratio) if PANEL_ART_AVAILABLE else (lambda d: d["area_ratio"]),
            reverse=True,
        )
        for i, d in enumerate(detections):
            if PANEL_ART_AVAILABLE: d.index = i
            else: d["index"] = i

        by_scale = {}
        for d in detections:
            s = d.scale if PANEL_ART_AVAILABLE else d["scale"]
            by_scale[s] = by_scale.get(s, 0) + 1

        with out_status:
            summary = ", ".join(f"{k}={v}" for k, v in sorted(by_scale.items()))
            print(f"  → {len(detections)} kept — {summary}\n")

        # # PRINT/OUTPUT DETECTION FILE
        # timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # path = Path(panel_path)
        # filename = path.parent / f"detections/{path.name}_{timestamp}.json"
        # print(f"Writing detections to file '{filename}'...")

        # with open(filename, "w") as f:
        #     json.dumps({
        #         "source_image_path": panel_path,
        #         "detections": [
        #             d.to_dict() if isinstance(d, Detection) else d
        #             for d in detections
        #         ],
        #         # "detection_idx_to_panel": detection_idx_to_panel
        #     }, f)
        
        # ── Annotated image ────────────────────────────────────────────────
        with out_image:
            fig_h = min(20, max(8, h / 80))
            fig_w = fig_h * (w / h)
            fig, ax = plt.subplots(figsize=(fig_w, fig_h))
            draw_detections(img_np, detections, ax)
            ax.set_title(f"{panel_path.name}  |  {len(detections)} detections", fontsize=10)
            plt.tight_layout()
            plt.show()

        # ── Detection table ────────────────────────────────────────────────
        with out_table:
            print(f"\n── {panel_path.name} ──────────────────────────────────────")
            print(f"{'#':>3}  {'scale':<10} {'area%':>6}  {'bbox (x,y,w,h)':>26}  "
                  f"{'pred_iou':>9}  {'stability':>10}")
            print("-" * 75)
            for d in detections:
                if PANEL_ART_AVAILABLE:
                    b = d.bbox
                    print(f"{d.index:>3}  {d.scale:<10} {d.area_ratio*100:>5.2f}%  "
                          f"({b['x']:>4},{b['y']:>4},{b['w']:>4},{b['h']:>4})  "
                          f"{d.predicted_iou:>9.4f}  {d.stability_score:>10.4f}")
                else:
                    b = d["bbox"]
                    print(f"{d['index']:>3}  {d['scale']:<10} {d['area_ratio']*100:>5.2f}%  "
                          f"({b[0]:>4},{b[1]:>4},{b[2]:>4},{b[3]:>4})  "
                          f"{d['predicted_iou']:>9.4f}  {d['stability_score']:>10.4f}")

        # ── Raw mask view ──────────────────────────────────────────────────
        if w_show_raw.value:
            with out_raw:
                fig_h2 = min(20, max(8, h / 80))
                fig_w2 = fig_h2 * (w / h)
                fig2, ax2 = plt.subplots(figsize=(fig_w2, fig_h2))
                ax2.imshow(img_np)
                font_size = max(5, min(w, h) / 120)
                lw = max(1, min(w, h) // 400)

                kept_bboxes = set()
                for d in detections:
                    if PANEL_ART_AVAILABLE:
                        b = d.bbox; kept_bboxes.add((b["x"], b["y"], b["w"], b["h"]))
                    else:
                        b = d["bbox"]; kept_bboxes.add(tuple(b))

                for m in raw_masks:
                    x, y, bw, bh = [int(v) for v in m["bbox"]]
                    ar = m["area"] / img_area
                    piou   = m["predicted_iou"]
                    stab_s = m["stability_score"]
                    asp = max(bw, bh) / max(min(bw, bh), 1) if min(bw, bh) > 0 else 999

                    if (x, y, bw, bh) in kept_bboxes:
                        colour, reason = "lime", "kept"
                    elif ar < mina:
                        colour, reason = "yellow", f"too small ({ar*100:.2f}%)"
                    elif ar > maxa:
                        colour, reason = "orange", f"too large ({ar*100:.1f}%)"
                    elif asp > masp:
                        colour, reason = "mediumpurple", f"aspect {asp:.1f}"
                    elif piou < iou_t or stab_s < stab:
                        colour, reason = "hotpink", f"quality iou={piou:.2f} stab={stab_s:.2f}"
                    else:
                        colour, reason = "lightgrey", "NMS"

                    ax2.add_patch(mpatches.Rectangle(
                        (x, y), bw, bh, linewidth=lw, edgecolor=colour, facecolor="none", alpha=0.7,
                    ))
                    ax2.text(x + 2, y + 10, reason, fontsize=font_size - 1,
                             color=colour, va="top", alpha=0.9)

                ax2.legend(handles=[
                    mpatches.Patch(color="lime",         label="kept"),
                    mpatches.Patch(color="yellow",       label="too small"),
                    mpatches.Patch(color="orange",       label="too large"),
                    mpatches.Patch(color="mediumpurple", label="bad aspect"),
                    mpatches.Patch(color="hotpink",      label="low quality (iou/stability)"),
                    mpatches.Patch(color="lightgrey",    label="NMS suppressed"),
                ], loc="upper right", fontsize=font_size, framealpha=0.8)
                ax2.set_title(
                    f"{panel_path.name} — {len(raw_masks)} raw SAM masks — colour = drop reason",
                    fontsize=10,
                )
                ax2.axis("off")
                plt.tight_layout()
                plt.show()

    # ── Shared parameter export ────────────────────────────────────────────
    with out_export:
        print("\n# ── Copy these values into panel_art/motif_segment.py ──")
        print(f"DEFAULT_IOU_THRESH       = {iou_t}")
        print(f"DEFAULT_STABILITY_THRESH = {stab}")
        print(f"DEFAULT_NMS_IOU          = {nms}")
        print(f"DEFAULT_MIN_AREA         = {mina}")
        print(f"DEFAULT_MAX_AREA         = {maxa}")
        print(f"DEFAULT_POINTS_PER_SIDE  = {pts}")
        print(f"DEFAULT_MAX_ASPECT       = {masp}")

run_btn.on_click(on_run)


['../../frobenius_artifacts/analysis/panels/EBA-B_00425_Ibadan_q97912_i1_panel_0_cropped.png', '../../frobenius_artifacts/analysis/panels/EBA-Div_00303_Ado_Ekiti_q166559_i1_panel_00.png', '../../frobenius_artifacts/analysis/panels/EBA-Div_00311_Ife_q166566_i1_panel_00.png', '../../frobenius_artifacts/analysis/panels/EBA-Div_00311_Ife_q166566_i1_panel_01.png', '../../frobenius_artifacts/analysis/panels/EBA-Div_00311_Ife_q166566_i1_panel_02.png', '../../frobenius_artifacts/analysis/panels/EBA-Div_00312_Ife_q166567_i1_panel_00.png', '../../frobenius_artifacts/analysis/panels/EBA-Div_00312_Ife_q166567_i1_panel_01.png', '../../frobenius_artifacts/analysis/panels/FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_00.png', '../../frobenius_artifacts/analysis/panels/FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_01.png', '../../frobenius_artifacts/analysis/panels/FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_02.png', '../../frobenius_artifacts/analysis/panels/FoA_04-5585_Modakeke_(Ife)_q48635_i1_panel_00.png

HTML(value="<h3 style='margin-bottom:4px'>SAM Parameter Tuning</h3>")

SelectMultiple(description='Panels:', index=(1, 4, 5, 6, 7, 8, 9, 12, 13, 14, 25), layout=Layout(width='70%'),…

HTML(value="<i style='color:#666'>11 selected — Ctrl/⌘+click to add, Shift+click to range-select</i>")

Output()

...

Output()

Output()

Output()

Output()

Output()

---
## points/side (8 / 16 / 32 / 64)
SAM works by placing a grid of probe points across the image and growing a mask from each one. More points → finer proposals, more
body-part-level fragments. Fewer points → coarser proposals biased toward whole figures and whole bands. 16 is the current sweet spot.
Go to 32 if you're missing small isolated symbols; go to 8 if you're getting too many fragments on dense knotwork.

---
## iou_thresh (0.40 – 0.95)
SAM's own internal confidence score for each mask it generates — higher means SAM is more certain the mask is a clean, well-bounded
object. The problem for carved wood is that large texture regions (a whole knotwork body, a full register band) score lower here because
their edges are ambiguous. Lower this to admit large carved regions; raise it to keep only clean, crisply-bounded shapes. Currently
0.70.

## stability (0.40 – 0.95)
Similar to iou_thresh but measures how consistent the mask is across small perturbations to the threshold. Large flat carved areas tend
to score lower. Lower = more permissive, same tradeoff as above. Currently 0.75.

▎ iou_thresh and stability both act as gates inside SAM before masks even reach the other filters. If you're getting zero or very few
▎ detections on a panel, lower these first.

---
## min_area (0.05% – 15% of panel area)
The smallest mask allowed through, expressed as a fraction of the whole panel. This is the primary lever against body-part fragments — a
hand or foot is typically 1–2% of a panel, a complete figure is 5%+. Raise it to suppress more fragments; lower it if you need to
capture small isolated symbols (e.g. a small Ifa board motif or a fine carved line detail at 0.1–0.5%). Currently 3%.

**Slider range extended**: previously 0.5%–15% (step 0.5%); now 0.05%–15% (step 0.05%) so you can reach the sub-1% region where
fine motifs live.

## max_area (30% – 100%)
The largest mask allowed. Prevents the degenerate "entire panel" blob that SAM sometimes generates. Also controls whether a whole
knotwork body (which can fill 80% of a narrow panel) registers as a register detection or gets filtered out. Currently 85% — the
knotwork register at 82% just barely passes.

---
## nms_iou (0.05 – 0.70)
Non-Maximum Suppression threshold. When two bounding boxes overlap by more than this fraction, the smaller one is dropped. Lower = more
aggressive suppression (fewer detections, less overlap between boxes). Higher = more permissive (adjacent motifs survive even if their
boxes touch). If you see duplicate boxes on the same figure, lower this. If adjacent figures are being suppressed, raise it.

---
## max_aspect (1.5 – 15.0)
Drops any mask whose bounding box is more elongated than this ratio (longer side ÷ shorter side). Filters out thin horizontal slivers
(e.g. cracks in the wood, border lines) that aren't meaningful motifs. Rarely needs touching — only lower it if you're getting many thin
strip detections, or raise it if you have very tall narrow panels where a legitimate figure has an extreme aspect ratio.
